# 4. start/stop과 완료 사건: 대화를 막지 않기

근거: [설계 글 §4 및 §3.4](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20).

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## async 함수와 전이중 상호작용의 차이
await slow_tool()만 하면 호출자는 여전히 결과를 기다립니다. start는 task ID를 즉시 반환하고, 완료는 큐의 새 사건으로 전달합니다. Foreground는 그 사이 다른 입력을 받습니다. 원문도 새 입력은 **현재 추론 step이 끝난 뒤** 처리한다고 설명하며, 토큰 생성 자체를 임의로 중단한다는 뜻은 아닙니다.

아래는 실제 asyncio Task를 사용합니다. 임의 sleep 시간으로 빠름을 주장하지 않고 Event 장벽을 사용해 작업이 끝나기 전에 다른 입력을 처리할 수 있음을 검증합니다.

In [1]:
import asyncio

class Jobs:
    def __init__(self): self.tasks={}; self.events=asyncio.Queue()
    def start(self,job_id,fn):
        if job_id in self.tasks: raise ValueError('중복 작업 ID')
        async def run():
            try:
                value=await fn()
                event={'id':job_id,'status':'completed','result':value}
            except Exception as exc:
                event={'id':job_id,'status':'failed','error':type(exc).__name__}
            return event
        def finished(task):
            # 실행 전 취소도 이 완료 경로에서 정확히 한 번 기록합니다.
            event={'id':job_id,'status':'cancelled'} if task.cancelled() else task.result()
            self.events.put_nowait(event)
        task=asyncio.create_task(run())
        self.tasks[job_id]=task
        task.add_done_callback(finished)
        return {'id':job_id,'status':'started'}
    async def stop(self,job_id):
        self.tasks[job_id].cancel()
        await asyncio.gather(self.tasks[job_id],return_exceptions=True)
    async def close(self):
        for task in self.tasks.values():
            if not task.done(): task.cancel()
        await asyncio.gather(*self.tasks.values(),return_exceptions=True)

jobs=Jobs(); entered=asyncio.Event(); release=asyncio.Event()
async def slow():
    entered.set(); await release.wait(); return '분석 완료'
try:
    receipt=jobs.start('analysis',slow)
    await asyncio.wait_for(entered.wait(),2)
    foreground_reply='새 지시를 접수했습니다'
    assert receipt['status']=='started' and not jobs.tasks['analysis'].done()
    print(foreground_reply)
    release.set()
    done=await asyncio.wait_for(jobs.events.get(),2)
    assert done['result']=='분석 완료'
    print(receipt,done)
finally:
    await jobs.close()

새 지시를 접수했습니다
{'id': 'analysis', 'status': 'started'} {'id': 'analysis', 'status': 'completed', 'result': '분석 완료'}


In [2]:
jobs=Jobs(); entered=asyncio.Event(); release=asyncio.Event()
try:
    jobs.start('cancel-me',slow)
    await asyncio.wait_for(entered.wait(),2)
    await jobs.stop('cancel-me')
    event=await jobs.events.get()
    assert event['status']=='cancelled'
    assert jobs.tasks['cancel-me'].done()
    print(event)
finally:
    await jobs.close()

{'id': 'cancel-me', 'status': 'cancelled'}


In [3]:
# 이벤트 루프에 실행 기회를 주기 전에 취소하는 경계 조건
jobs=Jobs()
async def never_started():
    raise AssertionError('이 함수는 실행되면 안 됩니다')
jobs.start('instant',never_started)
await jobs.stop('instant')
await jobs.close()
assert jobs.events.get_nowait()=={'id':'instant','status':'cancelled'}
assert jobs.events.empty()
assert all(task.done() for task in jobs.tasks.values())

jobs=Jobs()
async def fail(): raise ValueError('실습 오류')
jobs.start('failure',fail)
await asyncio.gather(*jobs.tasks.values())
await jobs.close()
assert jobs.events.get_nowait()=={'id':'failure','status':'failed','error':'ValueError'}
assert jobs.events.empty()
print('즉시 취소·실패도 종료 사건을 한 번 기록하고 정리합니다.')

즉시 취소·실패도 종료 사건을 한 번 기록하고 정리합니다.


## Speculative execution과 채널별 전달
먼저 재현·분석·초안 작성 같은 작업을 수행하고, 유용성이 확인될 때만 결과를 전달합니다. 아래는 외부 전송 없이 메시지 문자열만 생성합니다. Worker의 상세 로그는 내부에 두고 Router가 채널에 맞게 응답합니다. 100단어 제한은 원문이 IM에서 실험한 규칙이며 한국어 문자 수 규칙과 같지 않습니다.

In [4]:
def deliver(report,channel):
    if not report['verified']: return None
    if channel=='im': return report['summary']
    return f"문제: {report['summary']}\n검증: {report['evidence']}"
assert deliver({'verified':False},'im') is None
report={'verified':True,'summary':'회귀 원인을 재현했습니다.','evidence':'로컬 재현 테스트 통과'}
assert '검증:' in deliver(report,'github')
print(deliver(report,'im')); print(deliver(report,'github'))

회귀 원인을 재현했습니다.
문제: 회귀 원인을 재현했습니다.
검증: 로컬 재현 테스트 통과


## 직접 바꿔 보기

작업 실패도 성공과 같은 큐로 전달해 보세요. stop이 실제 외부 명령까지 중단했다고 말하려면 무엇을 더 검증해야 할까요?

## 해설과 참고 답안

fn에서 ValueError를 발생시키면 failed 사건이 나옵니다. asyncio Task 취소는 외부 프로세스 종료 보장이 아닙니다. 프로세스 그룹·원격 취소 요청·종료 확인·부분 결과 정리를 별도로 설계해야 합니다. 완료 작업 보존 기간과 큐 backpressure도 운영에 필요합니다.